# EDA on Gold layer data to identify what needs to be corrected or changed in Silver

- ERD modeling is done, however there is still an issue with the datasets country codes.
    - Reason: This dataset covers almost two centuries (1798 - 2022) and that includes alot of countries that doesn't really exist today(2026).\n This will need to be sorted or figured out

In [0]:
df_events = spark.sql("SELECT DISTINCT event_country AS code FROM marathos.silver.marathos_obt")
df_athletes = spark.sql("SELECT DISTINCT athlete_country AS code FROM marathos.silver.marathos_obt")

## Country codes & enforcement of rules

- Now that all Country codes has been read in and been assigned an alias, ill use the `union` set operator, deduplicate it and collect all country codes I need and get an LLM to compile a complete dataset that covers all countries in my dataset(both non-existing countries and existing)


- Result of the `union` set operator on all country codes shows that i still have more data to standardize.
    - Country code rules I am following is: All letters in CAPS.
    - Remove the one blank row
    - Change for example `sve` -> `SWE`

---

## Rules regarding Countries and Country codes.

I will follow the IOC standardisation rules for non-existing countries. If a runner was running for the soviet union or was born in the Soviet Union, I will keep that in my dataset and *not* change it to Russia(RUS).

- Sources + more IOC(International Olympic Committee) data:

```
When handling nations that no longer exist in the International Olympic Committee (IOC) database, the official best practice is to preserve historical data exactly as it was recorded at the time of the event, while assigning historical medals to the active successor states for all-time record keeping.  

Follow these primary guidelines for archival accuracy and data management:  

- Preserve Historical Context: Label statistics and historical leaderboards using the official name and IOC country code (e.g., URS for Soviet Union, YUG for Yugoslavia, TCH for Czechoslovakia) in effect during that specific Olympic Games.  

- Respect Unchanged Records: Do not strip past achievements from athletes; their historical gold, silver, and bronze finishes remain tied to their name and the exact nationality they competed under in their respective years.  

- https://ecda.eur.nl/en/portfolio/the-olympic-dream-a-definitive-database-of-every-olympian/  
```

In [0]:
df_all_codes = df_events.union(df_athletes).distinct().orderBy("code")
df_all_codes.display()

## Cell to check that my dim_countries.csv works as intended and exists

- Result of cell under:

| path | name | size | modification_time |
| :--- | :--- | :--- | :--- |
| /Volumes/marathos/default/raw/TWO_CENTURIES_OF_UM_RACES.csv | TWO_CENTURIES_OF_UM_RACES.csv | 827825781 | 1779540220000 |
| /Volumes/marathos/default/raw/dim_countries.csv | dim_countries.csv | 2988 | 1779983744000 | 

In [0]:
df_gold = spark.sql("SELECT * FROM marathos.silver.marathos_obt")

df_gold.limit().display()

In [0]:
%sql
-- Mart för insights relaterat till: HUR har ett land utvecklats över tid  när det kommer till sina events?
SELECT
    e.event_country,
    d.year, -- Vilka år hade flest events per land
    COUNT(DISTINCT e.event_id) AS total_events,
    COUNT(*) AS total_finishers
FROM marathos.gold.fact_results f
JOIN marathos.gold.dim_event e 
    ON f.event_id = e.event_id
JOIN marathos.gold.dim_date d  
    ON e.event_dates = d.date
GROUP BY 
    e.event_country,
    d.year
ORDER BY 
    total_events DESC;

In [0]:
%sql
-- Mart för att få insights över NÄR på året(säsong) det är "prime time" för events i ett land/region
-- Har tid på året någon korrelation med atleters prestation?
SELECT
    e.event_country,
    d.month,
    COUNT(DISTINCT e.event_id) AS total_events,
    COUNT(*) AS total_finishers,
    AVG(f.performance) AS avg_performance
FROM marathos.gold.fact_results f
JOIN marathos.gold.dim_event e 
    ON f.event_id = e.event_id
JOIN marathos.gold.dim_date d  
    ON e.event_dates = d.date
GROUP BY 
    e.event_country,
    d.month
ORDER BY 
    total_events DESC;